# SOA design exploration with JAX

**Design question:** which length and pump current give useful pulse gain without excessive pulse distortion or data-pattern memory?

A diode-driven semiconductor optical amplifier is a small nonlinear circuit with an internal gain reservoir. Here it becomes a workload for **compiled, batched, differentiable circuit simulation**. We build one Circulax circuit, use `jax.vmap` to solve a design grid, measure execution time with device synchronization, and differentiate a pulse-energy objective through the circuit transient.

The plotted data come from Circulax's DC and transient solvers. Python loops are used for plotting and the explicitly labeled serial timing baseline. Batched simulations are expressed as JAX transformations, without launching a simulator process per design. `vmap` exposes batch parallelism to XLA; actual execution depends on the CPU/GPU backend and is not a promise of one hardware thread per device.

The device parameters are illustrative. The main design study uses zero internal optical loss. Facet reflections, heating, and spectral gain dispersion are omitted. ASE is an output-noise approximation and does not deplete the reservoir.


In [ ]:
%matplotlib inline
import time
from functools import partial

import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import brentq

from circulax import compile_circuit
from circulax.components.base_component import PhysicsReturn, Signals, source
from circulax.components.electronic import Resistor, VoltageSource
from circulax.components.soa import SemiconductorOpticalAmplifier, ase_field, soa_scales, unsaturated_gain

plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.2, "figure.constrained_layout.use": True})
length_um = 600.0
reference_current_a = 0.080  # At the 600 µm reference length.
lifetime_s = 280e-12
alpha_h = 3.5
noise_figure_db = 6.5
wavelength_nm = 1540.0
bias_v = 1.8 * 0.02585 * np.log1p(reference_current_a / 1e-12) + 5 * reference_current_a
_, _, saturation_energy = map(float, soa_scales(wavelength_nm=wavelength_nm))
internal_saturation_power = saturation_energy / lifetime_s
print("JAX", jax.__version__, "| backend:", jax.default_backend(), "| devices:", jax.devices())
print(f"Reference: {length_um:g} µm, {reference_current_a * 1e3:g} mA, tau={lifetime_s * 1e12:g} ps")
print(f"Saturation energy {saturation_energy * 1e12:.3f} pJ")

## One circuit, many parameter sets

The electrical path is `anode → series resistance → diode || junction capacitance → cathode`. Conduction current pumps the gain reservoir; displacement current does not. The optical path maps input envelope to amplified output envelope. Diode saturation current and capacitance scale with contact area, and vertical series resistance scales inversely.

With active volume $V_a=wdL$, modal differential gain $a_m=\Gamma a$, and injection efficiency $\eta$,

$$h_0=a_mL\left(\frac{\eta I_D\tau_c}{qV_a}-N_{tr}\right),\qquad
\tau_c\dot h=h_0-h-\frac{P_{in}\tau_c}{E_{sat}}(e^h-1),\qquad
E_{sat}=\frac{h_P\nu wd}{a_m}.$$

The complex field is $A_{out}=A_{in}\exp[(1-j\alpha_H)h/2]$. Circulax represents the reservoir using a storage term in $F+dQ/dt=0$. The diode I–V relation drives the optical dynamics behaviorally; optical depletion does not feed back into electrical voltage.

A design vector contains **length (µm), pump-density multiplier, pulse energy (pJ), holding power (mW), lifetime (ps), Henry factor**. The multiplier is relative to 80 mA at 600 µm. Setting it to one holds contact current density fixed while length varies. Pump current scales with length; this prevents geometry changes from silently changing injection density. The transient uses a moving optical-envelope frame and omits transit delay.


In [ ]:
# A reproducible pattern includes long runs and random bits.
bits = np.r_[np.zeros(8, dtype=int), np.ones(8, dtype=int), np.random.default_rng(47).integers(0, 2, 80)]
samples_per_bit = 32
levels = np.where(bits, 1.0, 10 ** (-12 / 10))
pattern = gaussian_filter1d(np.repeat(levels, samples_per_bit), 2.5, mode="nearest")
pattern /= pattern.mean()
pattern_x = jnp.arange(pattern.size) / samples_per_bit
pattern_y = jnp.asarray(pattern)


@source(ports=("p1", "p2"), states=("i_src",), holomorphic=True)
def Envelope(
    signals: Signals,
    t: float,
    power: float = 2e-6,
    pulse_energy: float = 0.0,
    pulse_center: float = 400e-12,
    pulse_fwhm: float = 8e-12,
    bit_period: float = 125e-12,
    data: float = 0.0,
) -> PhysicsReturn:
    sigma = pulse_fwhm / jnp.sqrt(8 * jnp.log(2.0))
    pulse = pulse_energy / (sigma * jnp.sqrt(2 * jnp.pi)) * jnp.exp(-0.5 * ((t - pulse_center) / sigma) ** 2)
    modulation = (1 - data) + data * jnp.interp(t / bit_period, pattern_x, pattern_y)
    # A negligible power floor keeps sqrt derivatives finite in dark tails.
    field = jnp.sqrt(power * modulation + pulse + 1e-30)
    return {"p1": signals.i_src, "p2": -signals.i_src, "i_src": signals.p1 - signals.p2 - field}, {}


netlist = {
    "instances": {
        "SOA": {"component": "soa"},
        "BIAS": {"component": "bias"},
        "LIGHT": {"component": "light"},
        "LOAD": {"component": "load", "settings": {"R": 1.0}},
        "GND": {"component": "ground"},
    },
    "connections": {
        "GND,p1": ("BIAS,p2", "LIGHT,p2", "LOAD,p2", "SOA,cathode"),
        "BIAS,p1": "SOA,anode",
        "LIGHT,p1": "SOA,optical_in",
        "SOA,optical_out": "LOAD,p1",
    },
    "ports": {"input": "SOA,optical_in", "output": "SOA,optical_out"},
}
circuit = compile_circuit(
    netlist,
    {"soa": SemiconductorOpticalAmplifier, "bias": VoltageSource, "light": Envelope, "load": Resistor, "ground": lambda: 0},
    is_complex=True,
    backend="dense",
    rtol=1e-9,
    atol=1e-11,
)
base = {
    "BIAS.V": bias_v,
    "SOA.length_um": length_um,
    "SOA.lifetime_s": lifetime_s,
    "SOA.alpha_h": alpha_h,
    "SOA.saturation": 1.0,
    "SOA.wavelength_nm": wavelength_nm,
    "LIGHT.power": 2e-6,
    "LIGHT.pulse_energy": 0.0,
    "LIGHT.pulse_center": 400e-12,
    "LIGHT.pulse_fwhm": 12e-12,
    "LIGHT.bit_period": 125e-12,
    "LIGHT.data": 0.0,
}

In [ ]:
# The circuit topology is built once. Only numeric parameters change.
def design_params(design):
    length, density, energy_pj, hold_mw, lifetime_ps, henry = design
    contact_scale = length / 600.0
    current = reference_current_a * contact_scale * density
    voltage = 1.8 * 0.02585 * jnp.log1p(current / (1e-12 * contact_scale)) + 5 / contact_scale * current
    return {
        **base,
        "SOA.length_um": length,
        "BIAS.V": voltage,
        "SOA.lifetime_s": lifetime_ps * 1e-12,
        "SOA.alpha_h": henry,
        "LIGHT.power": hold_mw * 1e-3,
        "LIGHT.pulse_energy": energy_pj * 1e-12,
        "LIGHT.pulse_center": 60e-12,
        "LIGHT.pulse_fwhm": 8e-12,
    }


@partial(jax.jit, static_argnames=("duration", "step"))
def solve_design(design, *, duration=240e-12, step=0.25e-12):
    params = design_params(design)
    times = jnp.linspace(0.0, duration, int(round(duration / step)) + 1)
    initial = circuit.dc(params={**params, "LIGHT.pulse_energy": 0.0})
    sol = circuit.transient(
        t0=0.0, t1=duration, dt0=step, y0=initial, saveat=times, params=params, max_steps=int(round(duration / step)) + 16
    )
    return {
        "time": sol.ts,
        "input": circuit.port(sol.ys, "input"),
        "output": circuit.port(sol.ys, "output"),
        "h": jnp.real(circuit.port(sol.ys, "SOA,h")),
    }


def pulse_metrics(result):
    # Pulse experiments use zero holding power, so this is signal energy.
    pin, pout = jnp.abs(result["input"]) ** 2, jnp.abs(result["output"]) ** 2
    t = result["time"]
    ein, eout = jnp.trapezoid(pin, t), jnp.trapezoid(pout, t)
    centroid_in = jnp.trapezoid(t * pin, t) / ein
    centroid_out = jnp.trapezoid(t * pout, t) / eout
    return jnp.array([10 * jnp.log10(eout / ein), (centroid_out - centroid_in) * 1e12, eout * 1e12])


def evaluate(design):
    return pulse_metrics(solve_design(design))


reference_design = jnp.array([length_um, 1.0, 0.25, 0.0, lifetime_s * 1e12, alpha_h])
batched_solve = jax.jit(jax.vmap(solve_design))
batched_metrics = jax.jit(jax.vmap(evaluate))
print("Design columns: length_um, pump_density, energy_pJ, holding_mW, lifetime_ps, alpha_H")

## A design grid in one batched call

The three axes below vary length, pump density, and input pulse energy. Flattening the grid produces independent circuit simulations; `vmap` adds the batch axis while each circuit retains its own dynamic gain state. The maps compare useful energy gain with pulse-centroid advance. A large energy gain can come with strong reshaping, so gain alone is an incomplete design objective.


In [ ]:
lengths = jnp.linspace(320.0, 880.0, 8)
densities = jnp.linspace(0.82, 1.18, 8)
energies_pj = jnp.array([0.015, 0.15, 0.75])
ll, dd, ee = jnp.meshgrid(lengths, densities, energies_pj, indexing="ij")
designs = jnp.stack(
    [ll.ravel(), dd.ravel(), ee.ravel(), jnp.zeros(ll.size), jnp.full(ll.size, lifetime_s * 1e12), jnp.full(ll.size, alpha_h)],
    axis=1,
)
start = time.perf_counter()
metrics = batched_metrics(designs)
metrics.block_until_ready()
first_grid_seconds = time.perf_counter() - start
maps = np.asarray(metrics).reshape(len(lengths), len(densities), len(energies_pj), 3)
assert np.isfinite(maps).all()
print(f"{len(designs)} independent nonlinear transients; first call including compilation: {first_grid_seconds:.3f} s")

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey=True)
for col, energy in enumerate(energies_pj):
    for row, (metric_index, label) in enumerate([(0, "Energy gain (dB)"), (1, "Centroid shift (ps)")]):
        image = axes[row, col].pcolormesh(
            np.asarray(densities),
            np.asarray(lengths),
            maps[:, :, col, metric_index],
            shading="nearest",
            cmap="viridis" if row == 0 else "coolwarm",
        )
        fig.colorbar(image, ax=axes[row, col], label=label)
        axes[row, col].set_title(f"Input {float(energy):g} pJ")
        axes[row, col].set_xlabel("Pump-density multiplier")
    axes[0, col].set_ylabel("Length (µm)")
    axes[1, col].set_ylabel("Length (µm)")
plt.show()

# Check batch independence against individual circuit solves at distant grid points.
indices = np.array([0, len(designs) // 2, len(designs) - 1])
for idx in indices:
    np.testing.assert_allclose(metrics[idx], evaluate(designs[idx]), rtol=2e-7, atol=1e-8)
# Step refinement on the most strongly driven design.
refined = pulse_metrics(solve_design(designs[-1], step=0.125e-12))
np.testing.assert_allclose(metrics[-1, 2], refined[2], rtol=0.003)
print("Batch/scalar agreement and pulse-energy step refinement passed.")

## Measure compilation, dispatch, and batching separately

The benchmark uses identical designs, tolerances, time grids, and returned metrics. The serial baseline is a Python loop over an already compiled scalar solver; the batched path is `jit(vmap(...))`. Both paths synchronize before timing ends. Wrapper first-call timings include compilation and may reuse lower-level kernels from earlier cells; warmed timings reuse compiled executables. We report the median of three runs and also time the complete design grid after compilation.

A batching benefit can include reduced Python/device-dispatch overhead as well as vectorized numerical work. It is hardware- and workload-dependent; a small CPU workload may show little benefit. These are local measurements, not a comparison with another simulator. A new array shape can trigger another compilation.


In [ ]:
benchmark_designs = designs[jnp.linspace(0, len(designs) - 1, 24).astype(int)]
scalar_metrics = jax.jit(evaluate)
# A separate wrapper makes its first compilation visible in this benchmark.
benchmark_batch = jax.jit(jax.vmap(evaluate))

start = time.perf_counter()
scalar_metrics(benchmark_designs[0]).block_until_ready()
scalar_first = time.perf_counter() - start
start = time.perf_counter()
benchmark_batch(benchmark_designs).block_until_ready()
batch_first = time.perf_counter() - start


def measure(call, repeats=3):
    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        jax.block_until_ready(call())
        timings.append(time.perf_counter() - start)
    return float(np.median(timings))


serial_result = jnp.stack([scalar_metrics(d) for d in benchmark_designs])
batch_result = benchmark_batch(benchmark_designs)
np.testing.assert_allclose(serial_result, batch_result, rtol=2e-7, atol=1e-8)
serial_seconds = measure(lambda: jnp.stack([scalar_metrics(d) for d in benchmark_designs]))
batch_seconds = measure(lambda: benchmark_batch(benchmark_designs))
grid_seconds = measure(lambda: batched_metrics(designs))
print(f"Scalar first call: {scalar_first:.3f} s; {len(benchmark_designs)}-case batch first call: {batch_first:.3f} s")
print(f"Warm serial loop: {serial_seconds:.4f} s; warm batch: {batch_seconds:.4f} s")
print(f"Measured serial/batch ratio: {serial_seconds / batch_seconds:.2f}×")
print(f"Full warm grid: {len(designs)} cases in {grid_seconds:.4f} s ({len(designs) / grid_seconds:.1f} cases/s)")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(["Warm serial loop", "Warm JAX batch"], [serial_seconds, batch_seconds], color=["0.55", "tab:blue"])
ax.set(xlabel=f"Wall time for {len(benchmark_designs)} circuit transients (s)", title=str(jax.devices()[0]))
plt.show()

## Differentiate the design objective through the transient

Automatic differentiation gives local sensitivities of output pulse energy to geometry, pumping, input energy, and carrier lifetime. We use dimensionless design multipliers so the gradient components can be compared. Henry factor is fixed here because it changes phase rather than energy in this model. A central finite-difference check guards against a misleading gradient.

The objective below is $\log E_{out}$. The reported derivative is with respect to the logarithm of each design variable, i.e. $\partial\log E_{out}/\partial\log x$. These local derivatives are useful for optimization; they do not replace checking the broader nonlinear response maps.


In [ ]:
def objective(log_multipliers):
    # Keep holding power at zero. Scale L, pump density, input pulse energy, tau.
    scales = jnp.exp(jnp.array([log_multipliers[0], log_multipliers[1], log_multipliers[2], 0.0, log_multipliers[3], 0.0]))
    varied = reference_design * scales
    return jnp.log(evaluate(varied)[2])


value_and_gradient = jax.jit(jax.value_and_grad(objective))
value, gradient = value_and_gradient(jnp.zeros(4))
jax.block_until_ready(gradient)
epsilon = 2e-3
basis = jnp.eye(4) * epsilon
fd = jax.vmap(lambda delta: (objective(delta) - objective(-delta)) / (2 * epsilon))(basis)
np.testing.assert_allclose(gradient, fd, rtol=0.015, atol=0.003)
print("AD sensitivities:", np.asarray(gradient))
print("Finite differences:", np.asarray(fd))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(["Length", "Pump density", "Input energy", "Carrier lifetime"], gradient, color="tab:green")
ax.axhline(0, color="black", linewidth=0.8)
ax.set(ylabel="d log(output energy) / d log(parameter)", title="Gradient through Circulax's nonlinear transient")
plt.show()

## Gradient descent chooses a device length

Instead of maximizing gain until the length bound is reached, choose a finite **target output pulse energy**. Minimize the squared relative energy error plus a small footprint penalty:

$$J(L)=\left[E_{out}(L)/E_{target}-1\right]^2+0.02(L/600\,\mu m)^2.$$

The pump **current density** stays fixed, so a longer device also draws more total current. The length penalty slightly favors shorter devices over exact target matching. This is a soft design objective, not a proof of the shortest device satisfying a hard constraint. The allowable length range is 250–950 µm.

Two initial lengths run through projected gradient descent in parallel. `value_and_grad` differentiates through the full circuit transient; `lax.scan` compiles the optimizer loop, and `vmap` batches the two starts. A dense length sweep supplies an independent check of the final objective value.

In [ ]:
target_energy_pj = 3.0


def length_loss(normalized_length):
    design = reference_design.at[0].set(600 * normalized_length)
    output_energy = evaluate(design)[2]
    return (output_energy / target_energy_pj - 1) ** 2 + 0.02 * normalized_length**2


length_value_grad = jax.value_and_grad(length_loss)


def descend(initial_length):
    def step(x, _):
        loss, gradient = length_value_grad(x)
        return jnp.clip(x - 0.08 * gradient, 250 / 600, 950 / 600), jnp.array([x * 600, loss, gradient])

    final, history = jax.lax.scan(step, initial_length / 600, xs=None, length=36)
    return final * 600, history


optimize_lengths = jax.jit(jax.vmap(descend))
start = time.perf_counter()
optimized, history = optimize_lengths(jnp.array([330.0, 900.0]))
jax.block_until_ready(optimized)
print(f"Compiled optimization including first execution: {time.perf_counter() - start:.3f} s")
start = time.perf_counter()
jax.block_until_ready(optimize_lengths(jnp.array([330.0, 900.0])))
print(f"Warm 36-step optimization, two starts: {time.perf_counter() - start:.3f} s")
check_lengths = jnp.linspace(250, 950, 81)
check_losses = jax.jit(jax.vmap(length_loss))(check_lengths / 600)
final_losses = jax.vmap(length_loss)(optimized / 600)
np.testing.assert_allclose(optimized[0], optimized[1], atol=1.0)
assert np.all(np.asarray(final_losses) < np.min(check_losses) + 0.001)
assert np.all(np.asarray(final_losses) < np.asarray(history[:, 0, 1]))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for k, initial in enumerate([330, 900]):
    axes[0].plot(np.asarray(history[k, :, 0]), label=f"Start {initial} µm")
    axes[1].semilogy(np.asarray(history[k, :, 1]))
axes[0].set(xlabel="Gradient-descent iteration", ylabel="Length (µm)")
axes[0].legend()
axes[1].set(xlabel="Gradient-descent iteration", ylabel="Objective")
axes[2].plot(check_lengths, check_losses, label="Independent length sweep")
axes[2].scatter(optimized, final_losses, color="tab:red", label="Gradient descent")
axes[2].set(xlabel="Length (µm)", ylabel="Objective")
axes[2].legend()
plt.show()
for optimum in np.asarray(optimized):
    d = reference_design.at[0].set(optimum)
    print(f"Length {optimum:.2f} µm → output {float(evaluate(d)[2]):.3f} pJ; target {target_energy_pj:g} pJ")

## Inspect selected trajectories: amplitude, recovery, and phase

The grid summarizes many designs, but selected trajectories explain the mechanism. The first panel varies pulse energy at fixed geometry and pump density. The second varies holding power to expose the gain/recovery tradeoff. The third compares a zero-Henry-factor control with carrier-induced chirp; the fourth shows the associated spectra. Each family is evaluated with `vmap`.

We use the physical field convention Re{A(t) exp(−j2πν₀t)}. Thus physical frequency offset is −dφ/dt / (2π), and the FFT envelope frequency axis is negated. Chirp is checked against α_H dh/dt / (4π) only where the pulse has appreciable power.


In [ ]:
pulse_family = jnp.tile(reference_design, (3, 1)).at[:, 2].set(jnp.array([0.015, 0.15, 0.75]))
pulses = batched_solve(pulse_family)
recovery_family = jnp.tile(reference_design, (2, 1)).at[:, 3].set(jnp.array([0.002, 0.7]))
recovery_run = jax.jit(jax.vmap(lambda d: solve_design(d, duration=1.4e-9, step=1e-12)))
recovery = recovery_run(recovery_family)
phase_family = jnp.tile(reference_design, (3, 1)).at[:, 5].set(jnp.array([0.0, 2.0, 4.5]))
phase_runs = batched_solve(phase_family)
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for i, design in enumerate(np.asarray(pulse_family)):
    t = np.asarray(pulses["time"][i])
    p = np.abs(pulses["output"][i]) ** 2
    axes[0, 0].plot((t - 60e-12) * 1e12, p / p.max(), label=f"{design[2]:g} pJ")
axes[0, 0].set(xlim=(-18, 18), xlabel="Time from pulse center (ps)", ylabel="Peak-normalized output")
axes[0, 0].legend()
for i, hold in enumerate(np.asarray(recovery_family[:, 3])):
    axes[0, 1].plot(
        (np.asarray(recovery["time"][i]) - 60e-12) * 1e12,
        np.asarray(recovery["h"][i]) * 10 / np.log(10),
        label=f"Holding {hold:g} mW",
    )
axes[0, 1].set(xlim=(-30, 1200), xlabel="Time from pulse center (ps)", ylabel="Gain (dB)")
axes[0, 1].legend()
for i, henry in enumerate(np.asarray(phase_family[:, 5])):
    t = np.asarray(phase_runs["time"][i])
    field = np.asarray(phase_runs["output"][i])
    p = np.abs(field) ** 2
    mask = p > p.max() * 0.01
    chirp = -np.gradient(np.unwrap(np.angle(field)), t) / (2 * np.pi)
    check = henry * np.gradient(np.asarray(phase_runs["h"][i]), t) / (4 * np.pi)
    np.testing.assert_allclose(chirp[mask], check[mask], rtol=1e-7, atol=1.0)
    axes[1, 0].plot((t[mask] - 60e-12) * 1e12, chirp[mask] / 1e9, label=f"α_H = {henry:g}")
    frequencies = -np.fft.fftshift(np.fft.fftfreq(len(t), t[1] - t[0])) / 1e9
    spectrum = np.abs(np.fft.fftshift(np.fft.fft(field))) ** 2
    order = np.argsort(frequencies)
    axes[1, 1].plot(frequencies[order], (spectrum / spectrum.max())[order])
axes[1, 0].set(xlabel="Time from pulse center (ps)", ylabel="Physical frequency offset (GHz)")
axes[1, 0].legend()
axes[1, 1].set(xlim=(-200, 140), xlabel="Physical frequency offset (GHz)", ylabel="Normalized spectral density")
plt.show()

## Batched data-pattern stress test

A second batch tests data signals instead of isolated pulses. Rates and mean optical powers form a Cartesian grid, with 64 samples per bit at every rate. Every case receives the same reproducible bit pattern. We discard sixteen settling bits, then measure the logical-one power spread at bit centers. Eye diagrams show two selected cases; the map shows the full rate/power tradeoff.


In [ ]:
def data_case(rate_and_power):
    rate, power_mw = rate_and_power
    period = 1 / rate
    count = len(bits) * 64
    step = period / 64
    params = {
        **design_params(reference_design),
        "LIGHT.data": 1.0,
        "LIGHT.pulse_energy": 0.0,
        "LIGHT.bit_period": period,
        "LIGHT.power": power_mw * 1e-3,
    }
    initial = circuit.dc(params=params)
    ts = jnp.arange(count + 1) * step
    sol = circuit.transient(t0=0.0, t1=count * step, dt0=step, y0=initial, saveat=ts, params=params, max_steps=count + 16)
    p = jnp.abs(circuit.port(sol.ys, "output")) ** 2
    centers = p[jnp.arange(16, len(bits)) * 64 + 32]
    # Mask logical zeros without introducing dynamic-size arrays under JIT.
    ones_mask = jnp.asarray(bits[16:]) == 1
    low = jnp.min(jnp.where(ones_mask, centers, jnp.inf))
    high = jnp.max(jnp.where(ones_mask, centers, -jnp.inf))
    return p[16 * 64 : -1], 10 * jnp.log10(high / low)


rates = jnp.array([3e9, 6e9, 12e9])
mean_powers_mw = jnp.array([0.01, 0.05, 0.15, 0.5])
rr, pp = jnp.meshgrid(rates, mean_powers_mw, indexing="ij")
data_designs = jnp.stack([rr.ravel(), pp.ravel()], axis=1)
data_batch = jax.jit(jax.vmap(data_case))
data_power, one_spread = data_batch(data_designs)
jax.block_until_ready(one_spread)
assert np.isfinite(one_spread).all()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
for ax, index in zip(axes[:2], [2, 10]):
    p = np.array(data_power[index])
    p /= p.mean()
    for start in range(0, len(p) - 128, 64):
        ax.plot(np.arange(128) / 64, p[start : start + 128], color="tab:purple", alpha=0.14)
    ax.set(
        title=f"{float(data_designs[index, 0]) / 1e9:g} Gb/s, {float(data_designs[index, 1]):g} mW",
        xlabel="Time (unit intervals)",
        ylabel="Output / mean power",
    )
image = axes[2].imshow(np.asarray(one_spread).reshape(len(rates), len(mean_powers_mw)), aspect="auto", origin="lower", cmap="magma")
axes[2].set_xticks(range(len(mean_powers_mw)), [f"{float(p):g}" for p in mean_powers_mw])
axes[2].set_yticks(range(len(rates)), [f"{float(r) / 1e9:g}" for r in rates])
axes[2].set(xlabel="Mean input power (mW)", ylabel="Bit rate (Gb/s)")
fig.colorbar(image, ax=axes[2], label="One-level spread (dB)")
plt.show()
print(f"Solved {len(data_designs)} data-pattern cases in one JAX batch.")

## Does a long SOA need spatial segments?

Length alone does not decide this. With a single forward wave, linear material gain, uniform carrier lifetime, no passive loss and a moving time coordinate, integrating over length closes the gain equation. In that special case, a chain of cells has the same output as one integrated reservoir: stimulated-emission extraction telescopes as $\sum(P_{j+1}-P_j)=P_{out}-P_{in}$. Local carriers can still be nonuniform; one state simply cannot display their spatial profile.

With **distributed passive loss**, photon extraction from carriers is no longer the net output-minus-input power. Local recombination, nonuniform lifetime, independently varying pump sections, counterpropagating signals, and ASE depletion can also require spatial resolution. A traveling-wave model is needed when resolving propagation and bidirectional fields. See [Ó Dúill and Barry, *Improved Reduced Models for Single-Pass and Reflective Semiconductor Optical Amplifiers*](https://arxiv.org/abs/1407.5579) for the role of internal loss in subdivided SOA models.

Here each section has its own diode and gain state, and equal electrical bias maintains uniform contact current density. For section gain $h_j$ and passive attenuation exponent $\ell_j$, the field multiplier is $\exp[(h_j-\ell_j-j\alpha_Hh_j)/2]$. Carrier extraction is proportional to $h_j\,\mathrm{exprel}(h_j-\ell_j)P_{in,j}$, where $\mathrm{exprel}(x)=(e^x-1)/x$. This approximates constant gain within each section while distinguishing passive absorption from stimulated emission.

We compare 1, 2, 4, and 8 sections, holding total length, cross-section, current density, and timestep fixed. The lossless result is an invariant check; the lossy case is a mesh-convergence study. The final panel shows a longitudinal gain snapshot after depletion. **These cells have no propagation delay**: they resolve longitudinal carrier dynamics in the moving frame. At group index 3.6, a 1.4 mm device has about 17 ps transit time; add propagation delays/a traveling-wave treatment for laboratory-time arrival or rapidly varying pump signals. Segment count must be a static topology choice, so each topology is compiled separately.

In [ ]:
def segmented_fixture(sections):
    instances = {
        "BIAS": {"component": "bias"},
        "LIGHT": {"component": "light"},
        "LOAD": {"component": "load", "settings": {"R": 1.0}},
        "GND": {"component": "ground"},
    }
    for k in range(sections):
        instances[f"S{k}"] = {"component": "soa"}
    connections = {
        "GND,p1": tuple(["BIAS,p2", "LIGHT,p2", "LOAD,p2"] + [f"S{k},cathode" for k in range(sections)]),
        "BIAS,p1": tuple(f"S{k},anode" for k in range(sections)),
        "LIGHT,p1": "S0,optical_in",
        f"S{sections - 1},optical_out": "LOAD,p1",
    }
    connections.update({f"S{k},optical_out": f"S{k + 1},optical_in" for k in range(sections - 1)})
    return compile_circuit(
        {"instances": instances, "connections": connections, "ports": {"output": f"S{sections - 1},optical_out"}},
        {"soa": SemiconductorOpticalAmplifier, "bias": VoltageSource, "light": Envelope, "load": Resistor, "ground": lambda: 0},
        is_complex=True,
        backend="dense",
        rtol=1e-9,
        atol=1e-11,
    )


long_length = 1400.0
section_counts = [1, 2, 4, 8]
section_energy = {0.0: [], 18.0: []}
profiles = {}
for count in section_counts:
    segmented = segmented_fixture(count)

    @jax.jit
    def section_run(loss):
        params = {
            "length_um": long_length / count,
            "loss_db_per_cm": loss,
            "BIAS.V": bias_v,
            "lifetime_s": lifetime_s,
            "alpha_h": alpha_h,
            "LIGHT.power": 0.0,
            "LIGHT.pulse_energy": 0.3e-12,
            "LIGHT.pulse_center": 60e-12,
            "LIGHT.pulse_fwhm": 8e-12,
        }
        initial = segmented.dc(params={**params, "LIGHT.pulse_energy": 0.0})
        ts = jnp.linspace(0.0, 200e-12, 401)
        sol = segmented.transient(t0=0.0, t1=200e-12, dt0=0.5e-12, y0=initial, saveat=ts, params=params, max_steps=416)
        field = segmented.port(sol.ys, "output")
        gains = jnp.stack([jnp.real(segmented.port(sol.ys, f"S{k},h")) for k in range(count)], axis=1)
        return jnp.trapezoid(jnp.abs(field) ** 2, ts) * 1e12, ts, gains

    # Batch loss settings within each fixed topology.
    energy, ts, gains = jax.vmap(section_run)(jnp.array([0.0, 18.0]))
    for k, loss in enumerate([0.0, 18.0]):
        section_energy[loss].append(float(energy[k]))
    profiles[count] = (np.asarray(ts[1]), np.asarray(gains[1]))
np.testing.assert_allclose(section_energy[0.0], section_energy[0.0][0], rtol=0.003)
lossy_error_db = 10 * np.log10(np.asarray(section_energy[18.0]) / section_energy[18.0][-1])
assert abs(lossy_error_db[-2]) < 0.1
fig, axes = plt.subplots(1, 3, figsize=(13, 3.7))
for loss, values in section_energy.items():
    axes[0].plot(section_counts, values, "o-", label=f"Passive loss {loss:g} dB/cm")
axes[0].set(xlabel="Number of sections", ylabel="Output pulse energy (pJ)")
axes[0].legend()
axes[1].plot(section_counts, lossy_error_db, "o-")
axes[1].set(xlabel="Number of sections", ylabel="Energy difference from 8 sections (dB)")
ts, gain_profile = profiles[8]
for instant in [45e-12, 60e-12, 85e-12]:
    idx = np.searchsorted(ts, instant)
    axes[2].plot(
        (np.arange(8) + 0.5) * long_length / 8,
        gain_profile[idx] / (long_length / 8 * 1e-6),
        "o-",
        label=f"t = {instant * 1e12:g} ps",
    )
axes[2].set(xlabel="Position (µm)", ylabel="Active modal gain (1/m)")
axes[2].legend()
plt.show()
print("Lossy energy differences relative to 8 sections (dB):", lossy_error_db)
print("Transit time at n_g=3.6:", 3.6 * long_length * 1e-6 / 299792458 * 1e12, "ps")

## Numerical anchors and explicit noise bandwidth

Before using the batch results for design decisions, check the CW balance and the constant-gain limit. An independent scalar root verifies DC compression, and a short-pulse extraction law provides an energy benchmark. Exact 3.000 dB compression is used here, rather than the half-power approximation.

Output ASE is sampled with a fixed seed, outside the nonlinear residual. For one polarization, $S_{ASE}=n_{sp}\max(G-1,0)h_P\nu$ with the high-gain approximation $n_{sp}=10^{NF/10}/2$. Sample variance is $S_{ASE}/\Delta t$ over complex bandwidth $1/\Delta t$. A physical receiver filter is needed to obtain receiver-band noise. The noise check validates this stated bandwidth rather than interpreting timestep-wide noise as detected noise.


In [ ]:
ref_params = design_params(reference_design)
ref_current = reference_current_a * length_um / 600
h0 = float(unsaturated_gain(ref_current, length_um=length_um, lifetime_s=lifetime_s))
G0 = np.exp(h0)
input_powers = jnp.geomspace(2e-8, 6e-3, 32)
ops = circuit.dc(params={**ref_params, "LIGHT.pulse_energy": 0.0, "LIGHT.power": input_powers})
measured_h = np.real(circuit.port(ops, "SOA,h"))
roots = [brentq(lambda h, p=float(p): h - h0 + p / internal_saturation_power * np.expm1(h), 0.0, h0) for p in input_powers]
np.testing.assert_allclose(measured_h, roots, atol=2e-7)
linear = circuit.dc(
    params={**ref_params, "SOA.saturation": 0.0, "SOA.alpha_h": 0.0, "LIGHT.pulse_energy": 0.0, "LIGHT.power": 2e-6}
)
np.testing.assert_allclose(circuit.port(linear, "output"), np.sqrt(G0 * 2e-6), rtol=2e-7, atol=1e-10)
g3 = G0 * 10 ** (-3 / 10)
pout3 = internal_saturation_power * np.log(G0 / g3) / (g3 - 1) * g3
print(f"Reference weak-signal gain {10 * np.log10(G0):.2f} dB; 3 dB output compression {10 * np.log10(pout3 / 1e-3):.2f} dBm")

noise_dt = 0.75e-12
noise, expected = ase_field(
    np.full(100_000, G0), 1 / noise_dt, noise_figure_db=noise_figure_db, wavelength_nm=wavelength_nm, rng=np.random.default_rng(83)
)
np.testing.assert_allclose(np.mean(np.abs(noise) ** 2), expected.mean(), rtol=0.015)
fig, axes = plt.subplots(1, 2)
axes[0].semilogx(np.asarray(input_powers) * 1e3, measured_h * 10 / np.log(10))
axes[0].set(xlabel="CW input (mW)", ylabel="Gain (dB)")
noise_powers = np.abs(noise) ** 2
axes[1].hist(noise_powers / expected, bins=80, density=True, alpha=0.7, label="ASE samples")
x = np.linspace(0, 8, 300)
axes[1].plot(x, np.exp(-x), "--", label="Circular complex Gaussian power law")
axes[1].set(xlim=(0, 8), xlabel="Noise sample power / expected power", ylabel="Probability density")
axes[1].legend()
plt.show()
# Validate the short-pulse energy scale at the reference design.
reference_energy_out = float(evaluate(reference_design)[2])
u = float(reference_design[2]) * 1e-12 / saturation_energy
no_replenishment_pj = saturation_energy * np.log1p(G0 * np.expm1(u)) * 1e12
assert abs(reference_energy_out / no_replenishment_pj - 1) < 0.08
print("CW, linear limit, pulse-energy scale, batch independence, refinement, chirp, gradients, and ASE checks passed.")

The useful outputs of this notebook are the design maps, the synchronized performance measurements, and the sensitivities. Change the geometry or parameter grids and rerun to explore another design region. Change the device backend to assess CPU/GPU throughput, keeping compilation and warm execution separate. Larger grids can be processed in chunks if storing every waveform becomes memory-limited; returning only metrics reduces retained output, but does not remove all transient/adjoint memory costs.